# 07 · Productionisation

**Phase goal:** ship the winner as a single, portable artifact with a clean serving contract — and describe how it deploys.

## One Pipeline object = no train/serve skew
The shipped `.pkl` is a single sklearn `Pipeline` = `TargetEncoder` + model. The *exact* preprocessing fitted on the training data is re-applied at inference automatically, so there is zero chance of the encoding drifting between training and serving.

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import joblib, json
from car_pricing import config
pipe = joblib.load(config.PIPELINE_PATH)
mb = config.PIPELINE_PATH.stat().st_size/1024/1024
print(f'artifact: {config.PIPELINE_PATH.name}  ({mb:.2f} MB)')
print('steps   :', [s for s,_ in pipe.steps])

artifact: price_pipeline.pkl  (0.92 MB)
steps   : ['preprocess', 'model']


In [2]:
# The serving contract: give what you know, the rest is auto-filled
from car_pricing.predict import predict
for car in [{'make':'MARUTI','model':'SWIFT VXI','age':5,'km_driven':40000},
            {'make':'BMW','model':'X5','age':4},
            {'make':'HYUNDAI','model':'CRETA','fuel':'Diesel','transmission':'Automatic'}]:
    print(car, '->', predict(car))

{'make': 'MARUTI', 'model': 'SWIFT VXI', 'age': 5, 'km_driven': 40000} -> {'predicted_price_lakhs': 5.74, 'predicted_price_display': '₹5.74 Lakhs', 'price_band': 'Medium'}
{'make': 'BMW', 'model': 'X5', 'age': 4} -> {'predicted_price_lakhs': 18.68, 'predicted_price_display': '₹18.68 Lakhs', 'price_band': 'High'}
{'make': 'HYUNDAI', 'model': 'CRETA', 'fuel': 'Diesel', 'transmission': 'Automatic'} -> {'predicted_price_lakhs': 5.97, 'predicted_price_display': '₹5.97 Lakhs', 'price_band': 'Medium'}


In [3]:
meta = json.loads(config.METADATA_PATH.read_text())
print('brands       :', len(meta['makes_models']))
print('band edges   :', meta['band_edges'])
print('numeric dflts:', meta['numeric_defaults'])

brands       : 41
band edges   : [0.3, 3.99, 6.75, 20.9]
numeric dflts: {'km_driven': 52000.0, 'mileage': 19.3, 'engine': 1248.0, 'max_power': 86.8, 'age': 8.0}


## How it deploys (see [`docs/PIPELINE_DESIGN.md`](../docs/PIPELINE_DESIGN.md))
- **Batch:** `pipe.predict(df)` over a table — nightly repricing.
- **REST API:** wrap `car_pricing.predict.predict` in Flask/FastAPI — the sibling **`app_car_prices_flask`** shows the exact pattern (Docker + ECS).
- **Interactive:** a Streamlit UI — the sibling **`app_car_prices_streamlit`**.

## MLOps loop
- **CI:** `pytest` runs the data/feature/KPI-gate contracts on every push.
- **CT (continuous training):** re-run `python -m car_pricing.train` on new data; the KPI gate blocks a regressed model from shipping.
- **Monitoring:** track live MAE and input drift; retrain when either crosses a threshold.

The whole lifecycle — business KPI → data → features → model bake-off → evaluation → shippable pipeline — is now reproducible end to end.